# 03 Run ARF: reconcile legacy Delta against the BigQuery emulator

This is the same command you will run at the client. Only the settings file differs: `/lab/arf-config/settings.yaml` points at the local Spark and the emulator instead of Databricks and BigQuery.

ARF starts its own Spark session for this run. If memory is tight, shut down the kernels of notebooks 01 and 02 first (Kernel > Shut Down Kernel).

In [ ]:
!cd /arf && python -m arf check-config --settings /lab/arf-config/settings.yaml --tables /lab/arf-config/tables

In [ ]:
!cd /arf && python -m arf run --settings /lab/arf-config/settings.yaml --tables /lab/arf-config/tables -v

## The exceptions log from this run

The same file sits on your laptop under `arf-local-lab/output/<run_id>/arf_exceptions.csv`.

In [ ]:
from pathlib import Path

import pandas as pd

latest = max((p for p in Path("/lab/output").iterdir() if p.is_dir()), key=lambda p: p.name)
print("Run folder:", latest)
exceptions = pd.read_csv(latest / "arf_exceptions.csv")
exceptions[["table_id", "check_name", "column_name", "source_value", "target_value", "difference", "status", "message"]]

## What you should see

| Table | Expected | Why |
| --- | --- | --- |
| customers | PASS | Only agreed or harmless differences: the IST text became a UTC timestamp, and floats carry 1e-9 noise |
| orders | FAIL | 3 rows missing, 4 NULL customer ids, 2 amounts +5.00, `legacy_flag` dropped, `quantity` INT → STRING. Warnings: NOT NULL lost on `order_id`, `unit_price` scale 12 → 9, new column `load_ts`. The `row_hash` rows name every missing and changed order, including orders 600 and 900, whose status changed; no other check can see that |
| inventory | ERROR | The target table was never created; logged with a traceback, and the run carries on |

That is the same result as `python -m arf demo`, because both use the same data and the same planted defects.